# Notebook 32 — Vision-Language Models and Document Understanding

    ## Learning objectives

    - Understand processor inputs, image tokens, resolution, and memory costs
- Run a guarded Hugging Face image-text-to-text example
- Evaluate OCR, charts, spatial reasoning, and grounded structured output

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['huggingface-hub>=0.30,<1', 'python-dotenv>=1.1', 'pillow>=10']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 32.1 A multimodal request has two representations

A processor transforms images (resize/crop/normalize/patchify) and text (tokenize/chat
template) into model inputs. Visual encoders or native multimodal blocks create visual
tokens consumed by the language decoder. Higher resolution can preserve small text but
increases visual tokens, memory, prefill time, and cost. Inspect processor/model cards;
do not assume one image placeholder syntax fits every model.


In [ ]:
from PIL import Image, ImageDraw
image = Image.new("RGB", (640, 240), "white")
draw = ImageDraw.Draw(image)
draw.rectangle((50, 80, 590, 160), outline="navy", width=4)
draw.text((80, 105), "Invoice total: $123.45", fill="black")
display(image)


In [ ]:
# Optional remote inference; model availability varies by provider.
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")
if token:
    print("Use the current HF image-text-to-text model/provider from the model card.")
    print("Keep image bytes local unless you intend to send them to that provider.")
else:
    print("Remote example skipped: configure HUGGINGFACE_TOKEN.")


## 32.2 Documents are not just images

PDFs may contain extractable text, tables, reading order, vector graphics, scans, and
metadata. Prefer native extraction where reliable; use OCR/VLMs for visual structure and
scans; preserve page and bounding-box provenance. Prompt for a schema, validate it, and
retain evidence regions. Defend against visual prompt injection in screenshots/documents.


## 32.3 Evaluate by capability

Use exact field accuracy for extraction, normalized edit distance for OCR, table cell
metrics, bounding-box overlap for grounding, and expert labels for chart reasoning.
Slice by resolution, rotation, font size, handwriting, language, page count, and layout.
A fluent description can hide incorrect numbers—verify high-value fields deterministically.


## 32.4 Vision-language architecture families

Many VLMs encode images into patch features, project them into the language model's hidden
space, and place visual tokens alongside text tokens. Others use cross-attention or resampling
modules to compress visual features. Native multimodal models train integrated representations.
The processor decides resize, aspect handling, crops/tiles, normalization, placeholder count,
and chat formatting; model and processor revisions must match.

A square image with patch size p yields roughly `(H/p)*(W/p)` raw patches before model-specific
compression or tiling. Dynamic-resolution systems may create many more tokens for large or
unusual-aspect images. Multiple images multiply prefill/memory. Resize can erase small text;
aggressive tiling preserves it at cost. Report original/transformed dimensions and visual token
counts when debugging rather than assuming the model “saw” the source.


In [ ]:
# Patch-count planning intuition (actual processors may tile/compress differently).
import math
def patch_count(width, height, patch=14):
    return math.ceil(width/patch) * math.ceil(height/patch)
for width, height in [(224,224), (640,480), (1920,1080), (2480,3508)]:
    print(f"{width}x{height}: ~{patch_count(width,height):,} raw {14}x{14} patches")


## 32.5 OCR, layout, charts, and grounded extraction

OCR recognizes glyphs; document understanding also needs reading order, key-value association,
tables, page references, and visual hierarchy. Native PDF extraction is often cheaper and exact
for embedded text; OCR handles scans; layout models/VLMs interpret structure. A robust pipeline
combines them and retains page/bounding-box provenance. Compare extracted text to rendered pages
because hidden text layers can be wrong.

For charts, distinguish perception (axes, legend, marks, labels) from reasoning (trend, comparison,
calculation). Ask for structured fields with evidence coordinates or quoted labels. Validate
numeric units and recompute derivable quantities. For forms/invoices, define normalized schema,
required/optional fields, locale-aware dates/currency, confidence/routing, and page/region evidence.
Do not accept fluent summaries as proof of exact field extraction.


In [ ]:
# Example extraction schema and deterministic semantic validation.
from pydantic import BaseModel, Field, model_validator
class InvoiceExtraction(BaseModel):
    invoice_id: str
    currency: str = Field(pattern=r"^[A-Z]{3}$")
    subtotal: float = Field(ge=0)
    tax: float = Field(ge=0)
    total: float = Field(ge=0)
    evidence_page: int = Field(ge=1)
    @model_validator(mode="after")
    def totals_match(self):
        if abs((self.subtotal + self.tax) - self.total) > .02:
            raise ValueError("subtotal + tax does not match total")
        return self
print(InvoiceExtraction(invoice_id="INV-7", currency="USD", subtotal=100,
                        tax=23.45, total=123.45, evidence_page=1))


## 32.6 Multimodal prompting, batching, and serving

Follow the model's exact chat template and content-part structure. State which image each
instruction refers to; order images deterministically; avoid ambiguous “above/below.” Separate
user instructions from document content because images can contain prompt injection. Request a
schema and concise evidence, then validate. Cropping targeted regions can improve small-text
accuracy, but a cropper must not omit context; retain original coordinates.

Batch by compatible image size/token count to limit padding. Preprocessing can become CPU-bound;
profile decode, resize, OCR, transfer, prefill, and generation separately. Cache safe deterministic
preprocessing. Limit image count, pixels, file bytes, pages, decompression ratio, and processing
time. Strip or handle metadata intentionally. Serving support varies by vLLM/TGI/model version;
test exact modalities, templates, quantization, and concurrent memory.


In [ ]:
# Preserve crop provenance when zooming into a region.
box = (40, 70, 610, 175)  # left, top, right, bottom in original pixels
crop = image.crop(box)
display(crop.resize((crop.width*2, crop.height*2)))
provenance = {"original_size": image.size, "crop_box_xyxy": box,
              "scale_for_display": 2}
print(provenance)


## 32.7 Multimodal evaluation and safety reference

Build representative source files, not screenshots chosen for demos. Field extraction uses exact/
normalized accuracy and per-field precision/recall; OCR uses character/word error rate; tables use
structural/cell metrics; grounding uses IoU/pointing; charts use numeric tolerance and reasoning
correctness; descriptions require rubric/human review. Slice by resolution, font, scan quality,
rotation, handwriting, language, layout, page count, chart type, color dependence, and adversarial
text. Evaluate abstention when content is illegible or absent.

Threats include visual prompt injection, QR/links, steganographic or tiny text, malicious file
parsers, decompression bombs, metadata leakage, faces/biometrics, copyrighted/private documents,
and cross-tenant caches. Sandbox parsers, verify MIME/signature, cap resources, apply malware/DLP
policy, preserve access controls, and require confirmation before image-derived instructions cause
actions. The model should describe evidence; deterministic code should authorize effects.


## 32.8 Multimodal reference

| Task | Prefer/evaluate |
|---|---|
| Embedded PDF text | Native extraction plus rendered verification |
| Scanned text | OCR CER/WER and field accuracy |
| Forms/invoices | Structured schema, arithmetic/business validation, evidence region |
| Tables | Cell/structure accuracy, headers, merged cells |
| Charts | Axis/legend perception plus numeric reasoning tolerance |
| General images | Grounded descriptions, object/spatial slices |

Record source hash, page/image index, original/transformed dimensions, processor/model revision, crop/
tile coordinates, prompt, schema, and output. Evaluate illegible/missing-content abstention. Batch by
visual token load, not image count alone.

Files are untrusted complex inputs. Verify type/signature, sandbox parsers, cap bytes/pixels/pages/time/
decompression, enforce tenant ACL, and treat visible/hidden instructions as injection. Protect biometric,
private, copyrighted, and location-sensitive content according to policy. Validate high-value numeric
fields outside the model.


## Exercises

    1. Build a ten-image extraction set with field-level ground truth.
2. Compare native PDF text extraction with rendered-page VLM extraction.
3. Create a visual prompt-injection image and test the surrounding application controls.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
